# 🧠 Introduction to Prompt Engineering

> *"Prompting is not about tricking the AI — it's about being the clearest version of yourself."*

---

## 📌 Table of Contents

1. [Overview](#overview)
2. [Why This Lesson Matters](#why-this-lesson-matters)
3. [What is Prompt Engineering?](#what-is-prompt-engineering)
4. [The API vs. Chat Mindset](#the-api-vs-chat-mindset)
5. [Key Concepts](#key-concepts)
   - [Sensitivity to Input](#1-sensitivity-to-input)
   - [Structural Reproducibility with PromptTemplates](#2-structural-reproducibility-with-prompttemplates)
   - [Mitigation of AI Hallucinations](#3-mitigation-of-ai-hallucinations)
6. [Method Walkthrough](#method-walkthrough)
7. [Prompt Engineering in the Broader AI Context](#prompt-engineering-in-the-broader-ai-context)
8. [The House Analogy — Why the Foundation Matters](#the-house-analogy--why-the-foundation-matters)
9. [What You Will Build Toward](#what-you-will-build-toward)
10. [Summary & Key Takeaways](#summary--key-takeaways)

---

## Overview

This notebook is the **entry point** to a structured, 7-day prompt engineering curriculum. It introduces the core vocabulary, mental models, and practical tools you'll rely on throughout every subsequent lesson.

By the end of this notebook, you will be able to:

- Define prompt engineering and articulate why it matters
- Distinguish between "chatting" with an AI and *programming* with one
- Write and compare prompts that produce meaningfully different outputs
- Build a simple, reusable `PromptTemplate` using LangChain
- Explain why LLMs hallucinate and how prompt design can help prevent it

**Tools used in this notebook:** `openai` · `langchain` · `python-dotenv`

---

## Why This Lesson Matters

Most people's first instinct when using an LLM is to treat it like a search engine or a chatbot — type a question, read the answer, move on. That works for casual use. It doesn't work when you need *consistent, reliable, scalable* results.

This notebook reframes how you think about AI interaction entirely. Instead of asking:

> *"How do I get a good answer right now?"*

You start asking:

> *"How do I write an instruction that produces a good answer every single time — for any input?"*

That shift in perspective is the entire discipline of prompt engineering.

---

## What is Prompt Engineering?

**Prompt engineering** is the practice of designing, structuring, and refining the inputs you give to a language model in order to reliably produce a desired output.

It sits at the intersection of:

- **Linguistics** — the words you choose and how you arrange them
- **Logic** — the structure and clarity of your instructions
- **Computer Science** — the reproducibility, scalability, and testability of your approach

Think of it as writing a function for a very unusual kind of computer — one that takes natural language as its input and returns natural language as its output.

```
[Your Prompt]  →  [Language Model]  →  [Output]
```

The language model itself is a black box. Prompt engineering is the discipline of controlling what comes *out* by mastering what goes *in*.

### Why It's Important

| Without Prompt Engineering | With Prompt Engineering |
|---|---|
| Inconsistent outputs across runs | Reliable, reproducible results |
| Vague or off-topic responses | Precise, on-target outputs |
| Manual re-writing for every new input | Scalable templates that work for thousands of inputs |
| Unexpected hallucinations | Structured prompts that reduce confabulation |
| "Chatbot" mindset | Engineering mindset |

---

## The API vs. Chat Mindset

This tutorial introduces the **OpenAI API** and **LangChain** — and with them, a fundamental shift in how you should think about prompting.

### The Chat Mindset 💬

When most people use ChatGPT or Claude, they are in *conversation mode*:

- They type a message
- The AI responds
- They refine or follow up
- They copy the output they like

This is perfectly fine for personal use. But it doesn't scale. Every time you need an answer, you're starting from scratch. The prompt lives only in that session.

### The API Mindset ⚙️

When you work through an API, you start thinking differently:

- The prompt is **code** — it can be versioned, tested, and improved
- Variables replace hardcoded values — one template serves infinite inputs
- Output is **parsed and used downstream** — not just read by a human
- The goal is not *one good response* but a *system that produces good responses*

**A concrete example:**

| Chat Mindset | API Mindset |
|---|---|
| `"Write a product description for noise-cancelling headphones"` | `"Write a {tone} product description for {product_name} that emphasizes {key_feature}."` |
| Works once | Works for every product in your catalog |
| Lives in a browser tab | Lives in your codebase |

> 💡 **The mental model to hold onto:** You are not writing a message to an AI. You are writing a *reusable instruction* — a template — that should work reliably for 1,000 different inputs, model versions, and use cases.

---

## Key Concepts

### 1. Sensitivity to Input

One of the most important (and counterintuitive) properties of LLMs is how dramatically output changes with small changes to the input.

#### The Science

Language models work on **probability**. At each step, the model calculates the probability distribution over its entire vocabulary and selects the next most-likely word (or token). Your prompt is the starting condition that shapes that entire distribution.

Changing a single word doesn't just change *one* thing — it **shifts the entire mathematical trajectory** of everything the model generates after it.

```
"Explain quantum entanglement to a 10-year-old"
→  Simple analogies, accessible vocabulary, short sentences

"Describe quantum entanglement for a physics PhD student"
→  Mathematical notation, technical terms, deeper nuance
```

Same topic. Same model. Completely different output — because the probability landscape changed.

#### What This Means for You

This teaches two crucial habits:

1. **Patience and observation** — Don't assume the model is "broken" when it misses the mark. Assume *your instruction was ambiguous*.
2. **The engineering mindset** — Your job is to narrow the probability space so the model's most likely output is the one you actually want.

> 🧪 **Try it yourself:** Take any prompt and change just one word — swap `"list"` for `"explain"`, or `"professional"` for `"casual"`. Observe what changes. That observation *is* the skill.

---

### 2. Structural Reproducibility with PromptTemplates

In standard programming, you don't rewrite a function every time you call it. You write it *once*, define its parameters, and call it with different arguments.

Prompt engineering at scale requires the same discipline.

#### The Problem with Hardcoded Prompts

```python
# ❌ Brittle — works only once, for one thing
prompt = "Write a formal email to John about the Q3 budget delay."
```

#### The Solution: PromptTemplates

```python
# ✅ Reusable — works for any tone, recipient, or topic
from langchain.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["tone", "recipient", "topic"],
    template="Write a {tone} email to {recipient} about {topic}."
)

prompt = template.format(
    tone="formal",
    recipient="the finance team",
    topic="the Q3 budget delay"
)
```

#### Why This Changes Everything

| Hardcoded Prompt | PromptTemplate |
|---|---|
| Serves one specific case | Serves any combination of inputs |
| Must be rewritten manually | Updated in one place, applies everywhere |
| Hard to test systematically | Easy to run across a test suite |
| Lives in your head | Lives in your codebase |

> 💡 **The key insight:** This is the bridge between *hobbyist prompting* and *AI development*. Templates allow you to build systems, not just answers.

---

### 3. Mitigation of AI Hallucinations

#### What is Hallucination?

Language models are **next-token predictors**. They are trained to generate fluent, coherent text — not necessarily *true* text. When they don't know something, they don't say "I don't know." They generate the most *plausible-sounding* continuation of your prompt.

This produces **hallucinations**: confident, well-written, completely fabricated information.

> *LLMs are "confident liars" — not out of malice, but by design.*

#### How Prompt Engineering Helps

The framing of your question shapes *how* the model searches its learned patterns before generating a response.

```python
# ❌ Invites hallucination — model fills in gaps with guesses
"What was the exact verdict in Smith v. Jones (2019)?"

# ✅ Forces the model to surface uncertainty
"Do you have reliable information about the verdict in Smith v. Jones (2019)?
If not, say so clearly rather than guessing."
```

```python
# ❌ Encourages fabrication
"List five peer-reviewed studies proving X."

# ✅ Acknowledges the model's limitations
"Are there any well-known studies on X? If you're uncertain of exact citations,
describe the general research landscape instead."
```

#### The Bigger Picture

| Prompt Approach | Model Behavior |
|---|---|
| Vague, open-ended | Fills in gaps confidently — high hallucination risk |
| Specific, factual demands | May invent plausible-sounding "facts" |
| Permission to express uncertainty | More likely to hedge, qualify, or admit limits |
| Step-by-step reasoning required | Slows generation, catches errors mid-thought |

> 💡 **The key insight:** You cannot *eliminate* hallucination through prompting alone, but you can significantly *reduce* it by changing how much the model needs to invent in order to complete your request.

---

### Step 5 — Hallucination Mitigation Demo

```python
# Compare these two approaches
risky_prompt = "Cite three specific peer-reviewed papers about prompt engineering published in 2023."

safer_prompt = """Are there well-known academic works or research directions related to prompt engineering
from around 2022–2023? If you cannot recall specific verified citations, describe the general landscape
of research instead and flag which details you're uncertain about."""

print("RISKY PROMPT OUTPUT:")
print(get_completion(risky_prompt))

print("\n\nSAFER PROMPT OUTPUT:")
print(get_completion(safer_prompt))
```

> 🎯 **Goal:** See how giving the model permission to express uncertainty changes the quality and trustworthiness of the response.

---

## Prompt Engineering in the Broader AI Context

Prompt engineering doesn't exist in isolation. It sits inside a larger ecosystem of AI development practices:

```
┌─────────────────────────────────────────────────────────┐
│                    AI Application Layer                  │
│  ┌───────────────────────────────────────────────────┐  │
│  │            Prompt Engineering Layer               │  │
│  │  ┌─────────────────────────────────────────────┐  │  │
│  │  │         Language Model (LLM) Core           │  │  │
│  │  │  (GPT-4, Claude, Gemini, Llama, etc.)       │  │  │
│  │  └─────────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

Prompt engineering is the **interface layer** — the craft of communicating intent to a model that understands language but not *you*. It is relevant across:

- **Chatbots and assistants** — system prompts define personality and constraints
- **Content generation pipelines** — templates scale production across thousands of items
- **Data extraction and analysis** — structured prompts parse and categorize unstructured text
- **Code generation** — precise prompts produce functional, documented code
- **Multi-agent systems** — prompts coordinate the behavior of networks of AI agents

---

## The House Analogy — Why the Foundation Matters

> *"If you skip the basics and go straight to Chain-of-Thought, you might create a complex prompt that works — but you won't know why."*

Think of building prompt engineering skills like building a house:

```
┌────────────────────────────────────────────────────────┐
│  🏠  The Complete Structure                            │
├────────────────────────────────────────────────────────┤
│  Roof       │  Advanced Systems (RAG, Agents, CoT)     │
│  Walls      │  Chaining, Templates, Optimization       │
│  Plumbing   │  Evaluation, Safety, Ethics              │
│  Brickwork  │  Few-shot, Role Prompting, Reasoning     │
├────────────────────────────────────────────────────────┤
│  🪨 FOUNDATION  │  THIS NOTEBOOK                       │
│  Soil Test      │  What prompting is                   │
│  Concrete       │  How models interpret instructions   │
│  Rebar          │  Templates, sensitivity, hallucination│
└────────────────────────────────────────────────────────┘
```

Without a solid foundation:

- ❌ Your "advanced" prompts become **brittle** — they work today but break when the model updates
- ❌ You can't **debug** failures because you don't know what the model is responding to
- ❌ You can't **explain** why a technique works, so you can't adapt it to new problems

With a solid foundation:

- ✅ Every technique you learn has a **mental model** to attach to
- ✅ When something breaks, you know **where to look**
- ✅ You can **transfer** knowledge to new models, tools, and domains

## Setup

First, let's import the necessary libraries and configure the environment. We use python-dotenv to securely load API keys without hardcoding them into the notebook

In [10]:
import os
from getpass import getpass
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

A helper function (which is great for collaborative notebooks where a teammate might forget their .env), we prioritize the .env file first.

In [2]:
def setup_openai():
    load_dotenv() # Always try to load the file first
    
    api_key = os.getenv("OPENAI_API_KEY")
    
    if not api_key:
        print("🔑 OpenAI API Key not found in .env or System.")
        api_key = getpass("Enter your OpenAI API Key: ")
        os.environ["OPENAI_API_KEY"] = api_key
    else:
        print("✅ OpenAI API Key loaded from environment.")

In [3]:
# 1. Load the .env file
# This automatically puts variables into os.environ
load_dotenv() 

# 2. Check if the key exists (Fail Fast)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing! Check your .env file.")

# 3. Initialize (LangChain automatically looks for the env var)
llm = ChatOpenAI(model="gpt-4o-mini")

## Basic Concepts and Importance

At its core, prompt engineering is **how you talk to an AI so it actually listens**.

More formally: it is the practice of designing, structuring, and iterating on the inputs you give a language model to reliably produce a desired output. But that definition undersells it. Prompt engineering is not just phrasing — it is the discipline of understanding *how* a model interprets instructions and using that understanding to guide its behavior with precision.

Think of the language model as an extraordinarily well-read collaborator who has absorbed virtually all of human writing — but has no common sense, no memory of you, and no ability to ask clarifying questions unless you tell it to. Every time you send a prompt, you are giving that collaborator their *only* context. Your prompt is the entire briefing.

That's why it matters.

### Why Prompt Engineering Is a Core Skill

A language model's output is not random — it is a direct function of its input. The same model, given two different prompts on the same topic, can produce a response that is insightful or useless, accurate or hallucinated, concise or rambling. The model didn't change. The prompt did.

This means that **the quality of your output is largely the quality of your input** — and that is entirely within your control.

As AI becomes embedded in products, workflows, and decision-making pipelines, the ability to write prompts that work *consistently* — not just once, but at scale — becomes as fundamental as any other technical skill.

### A Simple Illustration

Before diving into structure and technique, let's ground this in a concrete example:

In [6]:
# 3. Use Message objects for better structure
messages = [HumanMessage(content="Explain prompt engineering in one sentence.")]

print(llm.invoke(messages).content)

Prompt engineering is the process of designing and refining the input prompts given to AI models to optimize their outputs for specific tasks or desired results.


Now, let's see how a more structured prompt can yield a more detailed response:

In [8]:
# 1. More modern way to create prompts (supports System/Human roles)
prompt = ChatPromptTemplate.from_template(
    "Provide a definition of {topic}, explain its importance, and list three key benefits."
)

# 2. The LCEL "Pipe" chain (this is the industry standard)
chain = prompt | llm 

# 3. Invoke with a simple dictionary
output = chain.invoke({"topic": "prompt engineering"})

print(output.content)

### Definition of Prompt Engineering

Prompt engineering is the process of designing and refining the inputs (prompts) given to AI language models to elicit desired responses or improve the quality of the output generated. This practice involves formulating questions, instructions, or context in a way that guides the model toward producing relevant, accurate, and useful information. Prompt engineering is essential for maximizing the effectiveness of AI models, particularly in applications like natural language processing, machine translation, and content generation.

### Importance of Prompt Engineering

Prompt engineering is crucial because the quality of outputs generated by AI models is highly dependent on the inputs they receive. By carefully crafting prompts, users can significantly enhance the model's performance, reduce ambiguities, and minimize the potential for generating misleading or irrelevant responses. As AI systems are increasingly integrated into various industries and 

### Why Prompt Engineering Matters

The way you phrase a request doesn't just change the *style* of the response — it changes the *substance*. A well-engineered prompt can be the difference between an AI that hallucinates confidently and one that reasons carefully. Between a response that's technically correct and one that's actually useful.

Here's what prompt engineering gives you control over:

**Output quality and relevance.** A vague prompt forces the model to guess what you want. A precise one narrows the probability space so the model's best guess is also your desired outcome. Specificity is not pedantry — it is how you eliminate the gap between what the AI generates and what you actually need.

**Task performance.** Language models are generalists. Without guidance, they default to a generic middle ground. A well-structured prompt effectively specializes the model for your task — whether that's legal summarization, code review, or customer support — without any retraining required.

**Bias and limitation mitigation.** Models inherit the patterns and gaps in their training data. Prompt design can't fix the model, but it can compensate — by asking the model to reason step by step before answering, to acknowledge uncertainty, or to consider multiple perspectives before drawing a conclusion.

**Audience and tone adaptation.** The same underlying content should read differently for a PhD researcher, a first-year student, and a non-technical executive. Prompt engineering makes that adaptation systematic rather than manual, letting one template serve every audience through variable substitution.

The following examples demonstrate all four of these dimensions in action — same topic, different prompts, meaningfully different results:

In [9]:
# 1. Prepare your inputs as a list of strings (or dictionaries if using a Template)
prompts = [
    "List 3 applications of AI in healthcare.",
    "Explain how AI is revolutionizing healthcare, with 3 specific examples.",
    "You are a doctor. Describe 3 ways AI has improved your daily work in the hospital."
]

# 2. Use .batch() instead of a loop
# This handles the concurrency for you automatically
responses = llm.batch(prompts)

# 3. Print the results
for i, response in enumerate(responses, 1):
    print(f"\nResponse {i}:")
    print(response.content)
    print("-" * 50)


Response 1:
Here are three applications of AI in healthcare:

1. **Clinical Decision Support**: AI systems are used to analyze patient data and assist healthcare professionals in making more informed decisions. These systems can provide recommendations for diagnosis, treatment options, and medication management based on a patient’s medical history, symptoms, and current research.

2. **Medical Imaging and Diagnostics**: AI algorithms, particularly those based on deep learning, are employed to analyze medical images such as X-rays, MRIs, and CT scans. They can detect abnormalities (e.g., tumors, fractures) with high accuracy, allowing for earlier diagnosis and intervention.

3. **Predictive Analytics for Patient Outcomes**: AI can analyze large datasets from electronic health records (EHRs) to identify patterns and predict patient outcomes. This can help in risk stratification, enabling healthcare providers to proactively manage patients who are at higher risk for complications or read

### Understanding the Results

These three responses illustrate something fundamental: the model didn't change — your instructions did. Each prompt acted as a different lens, and the AI shifted its perspective, depth, and language accordingly.

---

**Response 1 — The Generalist**
This was a basic, unqualified request. With no persona, no audience, and no scope defined, the model defaulted to a "textbook" mode: broad categories, safe generalizations, no specifics. It isn't wrong — it's just optimized for *nobody in particular*, which means it's only marginally useful for anybody. This is the AI's neutral gear.

**Response 2 — The Detailed Analyst**
Two small changes — the word *"revolutionizing"* and the phrase *"specific examples"* — shifted the output entirely. "Revolutionizing" signals that you want impact and momentum, not a neutral overview. "Specific examples" closes the door on vague generalities and forces the model to surface concrete evidence: named companies, real systems, measurable outcomes. The result reads like a professional briefing rather than an encyclopedia entry. This is what *contextual specificity* buys you.

**Response 3 — The Domain Expert**
Assigning the persona *"You are a doctor"* did more than change the tone — it changed the entire frame of reference. The model stopped describing what the technology *does* and started describing what it *feels like to use it*. Terms like burnout, patient safety, and administrative load entered the response because those are the concerns that live inside a clinician's perspective. The same facts, reprocessed through a different identity, produced a human-centered narrative that a purely technical prompt never would have reached.

---

The pattern across all three is the same: **the model reflects the assumptions embedded in your prompt**. A vague prompt returns a generic response. A specific, framed, persona-driven prompt returns something that feels almost authored. That gap between the two is exactly where prompt engineering lives.

## Role in AI and Language Models

Language models are remarkable but imperfect instruments. They are trained on vast corpora of human text, which makes them fluent, knowledgeable, and remarkably capable — but that same training also makes them prone to overconfidence, generalization, and blind spots. They don't know what they don't know. They can't ask you what you meant. And without guidance, they optimize for *plausibility* rather than *precision*.

Prompt engineering is the discipline that bridges the gap between what a model *can* do and what it *actually does* in practice.

**Tailoring outputs to specific needs.** A language model out of the box is a generalist. It has no default understanding of your industry, your audience, your constraints, or your definition of a "good" answer. Prompt engineering provides that context explicitly — transforming a general-purpose model into something that behaves like a specialist for your exact use case, without any retraining or fine-tuning required.

**Improving accuracy and relevance.** Models don't retrieve facts the way a search engine does — they *generate* text that is statistically consistent with their training. That process can produce responses that sound authoritative but are subtly (or dramatically) wrong. Prompts that require step-by-step reasoning, source acknowledgment, or explicit uncertainty force the model to slow down its generation process and produce responses that are more grounded and less likely to drift into confabulation.

**Enabling complex task completion.** A single, monolithic prompt asking a model to perform a complex multi-part task will almost always underperform. Prompt engineering introduces techniques like task decomposition — breaking a complex objective into a sequence of smaller, well-defined steps — that bring genuinely difficult problems within reach. The model isn't smarter, but the structure you provide allows it to apply its capabilities more effectively at each stage.

**Reducing bias and improving fairness.** Models inherit the patterns embedded in their training data, including its biases, blind spots, and overrepresentations. Prompts can't rewrite a model's weights — but they can counteract specific tendencies. Asking a model to argue multiple sides before concluding, to flag assumptions, or to consider underrepresented perspectives doesn't fix the underlying model, but it meaningfully shifts the output toward something more balanced and considered.

The examples that follow demonstrate these roles directly — showing how the same model limitation can produce two very different outcomes depending on whether the prompt is designed to work with the model's tendencies or against them:

In [13]:
# 1. Defined with specific roles (System vs Human)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that evaluates statements for factual accuracy."),
    ("human", "Evaluate this statement: {statement}")
])

# 2. Add the Output Parser to the end of the pipe
# This removes the need for .content later
chain = prompt | llm | StrOutputParser()

# 3. Invoke directly to get the string
result = chain.invoke({"statement": "The capital of France is London."})
print(result)

The statement is false. The capital of France is Paris, not London.


### Improving Complex Problem-Solving

A language model asked to solve a complex problem in a single pass will almost always cut corners. Not because it lacks the knowledge, but because its generation process is linear — each word is predicted from the last, which means the model commits to a direction early and rarely backtracks. On simple tasks, this works fine. On multi-step problems, it produces answers that are fluent but shallow, skipping logical steps the model never explicitly worked through.

The fix is structural. When you engineer a prompt that forces the model to reason *out loud* — to break the problem into stages, address each one in sequence, and only then draw a conclusion — you are fundamentally changing how it arrives at an answer. The model isn't smarter, but the scaffold you've built makes it harder to skip steps and easier to catch its own errors along the way.

This is the principle behind techniques like Chain-of-Thought prompting, which you'll explore in depth later in this curriculum. But even before those formal techniques, the instinct to *decompose before answering* is one of the highest-leverage habits you can build.

The following example demonstrates the difference between asking for an answer and asking for a process:

In [19]:
from langchain_core.prompts import PromptTemplate

problem_solving_prompt = PromptTemplate(
    input_variables=["problem"],
    template="""Solve the following problem step by step:
    Problem: {problem}
    Solution:
    1)"""
)

chain = problem_solving_prompt | llm
print(chain.invoke("Calculate the compound interest on $1000 invested for 5 years at an annual rate of 5%, compounded annually.").content)

To calculate the compound interest on an investment using the formula for compound interest, follow these steps:

### Step 1: Identify the variables
- Principal (P): $1000 (the initial amount)
- Rate (r): 5% per year (in decimal form, this is 0.05)
- Time (t): 5 years
- Compounding frequency (n): Since it is compounded annually, n = 1.

### Step 2: Use the compound interest formula
The formula for compound interest is given by:
\[
A = P \left(1 + \frac{r}{n}\right)^{nt}
\]
Where:
- \(A\) is the amount of money accumulated after n years, including interest.
- \(P\) is the principal amount (the initial amount).
- \(r\) is the annual interest rate (decimal).
- \(n\) is the number of times that interest is compounded per unit year.
- \(t\) is the time in years.

### Step 3: Substitute the values into the formula
In our case, we substitute \(P = 1000\), \(r = 0.05\), \(n = 1\), and \(t = 5\).
\[
A = 1000 \left(1 + \frac{0.05}{1}\right)^{1 \times 5}
\]
This simplifies to:
\[
A = 1000 \left(1

In [14]:
# 1. Cleaner template creation
# Using ChatPromptTemplate is better for "gpt-4o-mini"
prompt = ChatPromptTemplate.from_template(
    "Solve the following problem step by step:\nProblem: {problem}\nSolution:\n1)"
)

# 2. Complete the chain with an Output Parser
# This ensures 'chain.invoke' returns the string directly
chain = prompt | llm | StrOutputParser()

# 3. Execution
problem_text = "Calculate the compound interest on $1000 invested for 5 years at an annual rate of 5%, compounded annually."
print(f"1) {chain.invoke({'problem': problem_text})}")

1) To calculate the compound interest on an investment, we can use the compound interest formula:

\[
A = P(1 + r)^n
\]

where:
- \( A \) = the amount of money accumulated after n years, including interest.
- \( P \) = the principal amount (the initial amount of money).
- \( r \) = annual interest rate (decimal).
- \( n \) = number of years the money is invested or borrowed.

Let's proceed step by step for the given problem:

1. **Identify the values**:
   - Principal amount \( P = 1000 \)
   - Annual interest rate \( r = 5\% = 0.05 \)
   - Number of years \( n = 5 \)

2. **Plug the values into the formula**:
   \[
   A = 1000(1 + 0.05)^5
   \]

3. **Calculate \( (1 + r) \)**:
   \[
   1 + 0.05 = 1.05
   \]

4. **Raise \( 1.05 \) to the power of \( n \) (which is 5)**:
   \[
   (1.05)^5
   \]

   Calculate \( (1.05)^5 \):
   - \( 1.05^2 = 1.1025 \)
   - \( 1.05^3 = 1.1025 \times 1.05 = 1.157625 \)
   - \( 1.05^4 = 1.157625 \times 1.05 = 1.21550625 \)
   - \( 1.05^5 = 1.21550625 \times 